# Análisis Económico: Mercado Inmobiliario ZonaProp

Dataset: Casas, departamentos y PH en venta - Capital Federal

**Objetivo**: Entender el mercado inmobiliario desde una perspectiva económica.

## 1. Cargamos los datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar el estilo de los gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Cargar el CSV (utf-8-sig quita el BOM; low_memory=False evita tipos mixtos por columna)
df = pd.read_csv(
    '../data/casas-departamentos-ph-venta-capital-federal.csv',
    encoding='utf-8-sig',
    low_memory=False,
)

# Mostrar todas las columnas al imprimir tablas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print(f"Total de propiedades: {len(df):,} | Columnas: {df.shape[1]}")
print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nPrimeras filas (columnas clave):")
columnas_clave = ['precio_usd', 'precio_m2_usd', 'barrio', 'tipo', 'ambientes',
                  'dormitorios', 'banos', 'superficie_total', 'superficie_cubierta']
df[columnas_clave].head()

## 2. Limpieza y preparación de datos

In [ ]:
# Revisar tipos de datos y valores faltantes
print("Valores faltantes por columna:")
print(df.isnull().sum())

print(f"\nTipos de datos:")
print(df.dtypes)

In [ ]:
# Crear una versión limpia para análisis
# Mantener solo propiedades con precio en USD y superficie cubierta
df_analisis = df[
    (df['precio_usd'].notna()) & 
    (df['superficie_cubierta'].notna()) &
    (df['precio_usd'] > 0) &
    (df['superficie_cubierta'] > 0)
].copy()

print(f"Propiedades con datos completos: {len(df_analisis)} de {len(df)}")
print(f"Propiedades excluidas: {len(df) - len(df_analisis)}")

## 3. ESTADÍSTICAS GENERALES DE PRECIOS

In [ ]:
# Estadísticas básicas de precios
print("=" * 60)
print("PRECIOS EN USD")
print("=" * 60)

stats = {
    'Mínimo': df_analisis['precio_usd'].min(),
    'Q1 (25%)': df_analisis['precio_usd'].quantile(0.25),
    'Mediana (50%)': df_analisis['precio_usd'].median(),
    'Promedio (Media)': df_analisis['precio_usd'].mean(),
    'Q3 (75%)': df_analisis['precio_usd'].quantile(0.75),
    'Máximo': df_analisis['precio_usd'].max(),
    'Desviación Estándar': df_analisis['precio_usd'].std(),
}

for metrica, valor in stats.items():
    print(f"{metrica:.<40} USD ${valor:>15,.0f}")

# Rango intercuartílico (diferencia entre Q3 y Q1)
riq = stats['Q3 (75%)'] - stats['Q1 (25%)']
print(f"\n{'Rango Intercuartílico (IQR)':.<40} USD ${riq:>15,.0f}")

# Coeficiente de variación (cuán dispersos están los precios)
cv = (stats['Desviación Estándar'] / stats['Promedio (Media)']) * 100
print(f"{'Coef. Variación (dispersión)':.<40} {cv:>19.1f}%")

## 4. PRECIOS POR METRO CUADRADO

In [ ]:
# Calcular precio por m² (para propiedades que tengan ese dato)
df_analisis['precio_m2_calculado'] = df_analisis['precio_usd'] / df_analisis['superficie_cubierta']

print("=" * 60)
print("PRECIO POR METRO CUADRADO (USD/m²)")
print("=" * 60)

stats_m2 = {
    'Mínimo': df_analisis['precio_m2_calculado'].min(),
    'Mediana': df_analisis['precio_m2_calculado'].median(),
    'Promedio': df_analisis['precio_m2_calculado'].mean(),
    'Máximo': df_analisis['precio_m2_calculado'].max(),
}

for metrica, valor in stats_m2.items():
    print(f"{metrica:.<40} USD ${valor:>12,.0f}/m²")

## 5. ANÁLISIS POR TIPO DE PROPIEDAD

In [ ]:
# Agrupar por tipo de propiedad
print("=" * 60)
print("ANÁLISIS POR TIPO DE PROPIEDAD")
print("=" * 60)

por_tipo = df_analisis.groupby('tipo').agg({
    'precio_usd': ['count', 'mean', 'median', 'min', 'max'],
    'precio_m2_calculado': 'mean',
    'superficie_cubierta': 'mean',
    'ambientes': 'mean',
    'dormitorios': 'mean'
}).round(0)

print(por_tipo)

# Vista más legible
print("\n" + "="*60)
for tipo in df_analisis['tipo'].unique():
    datos = df_analisis[df_analisis['tipo'] == tipo]
    print(f"\n{tipo.upper()}")
    print(f"  Cantidad: {len(datos)}")
    print(f"  Precio promedio: USD ${datos['precio_usd'].mean():,.0f}")
    print(f"  Precio mediano: USD ${datos['precio_usd'].median():,.0f}")
    print(f"  Precio/m² promedio: USD ${datos['precio_m2_calculado'].mean():,.0f}")
    print(f"  Superficie promedio: {datos['superficie_cubierta'].mean():.0f} m²")
    print(f"  Ambientes promedio: {datos['ambientes'].mean():.1f}")

## 6. ANÁLISIS POR BARRIO (TOP 15)

In [ ]:
# Filtrar barrios con al menos 10 propiedades
barrios_minimo = df_analisis['barrio'].value_counts()
barrios_validos = barrios_minimo[barrios_minimo >= 10].index.tolist()

df_barrios = df_analisis[df_analisis['barrio'].isin(barrios_validos)]

print("=" * 80)
print("PRECIOS POR BARRIO (solo barrios con 10+ propiedades)")
print("=" * 80)

por_barrio = df_barrios.groupby('barrio').agg({
    'precio_usd': ['count', 'mean', 'median', 'min', 'max'],
    'precio_m2_calculado': 'mean',
}).sort_values(('precio_usd', 'mean'), ascending=False)

print(por_barrio.head(15))

# Vista más legible: Top 10 barrios más caros
print("\n" + "="*80)
print("TOP 10 BARRIOS MÁS CAROS (precio promedio)")
print("="*80)

top_caros = df_barrios.groupby('barrio').agg({
    'precio_usd': ['count', 'mean'],
    'precio_m2_calculado': 'mean'
}).sort_values(('precio_usd', 'mean'), ascending=False).head(10)

for barrio, row in top_caros.iterrows():
    cant = int(row[('precio_usd', 'count')])
    precio = row[('precio_usd', 'mean')]
    precio_m2 = row[('precio_m2_calculado', 'mean')]
    print(f"{barrio:.<30} ${precio:>12,.0f} (${precio_m2:>7,.0f}/m² | {cant} props)")

## 7. RELACIÓN ECONOMICA: AMBIENTES vs PRECIO

In [ ]:
# Análisis por número de ambientes
df_amb = df_analisis[df_analisis['ambientes'].notna() & (df_analisis['ambientes'] > 0)]

print("=" * 60)
print("PRECIOS POR CANTIDAD DE AMBIENTES")
print("=" * 60)

por_ambientes = df_amb.groupby('ambientes').agg({
    'precio_usd': ['count', 'mean', 'median'],
    'precio_m2_calculado': 'mean',
    'superficie_cubierta': 'mean'
}).sort_index()

print(por_ambientes)

# Vista más clara
print("\n" + "="*60)
for amb in sorted(df_amb['ambientes'].unique()):
    if amb <= 6:  # Solo mostramos hasta 6 ambientes
        datos = df_amb[df_amb['ambientes'] == amb]
        print(f"\n{int(amb)} AMBIENTE(S): {len(datos)} propiedades")
        print(f"  Precio promedio: USD ${datos['precio_usd'].mean():>12,.0f}")
        print(f"  Superficie promedio: {datos['superficie_cubierta'].mean():>10.0f} m²")
        print(f"  Precio por ambiente: USD ${datos['precio_usd'].mean() / amb:>10,.0f}")

## 8. EXPENSAS: COSTO MENSUAL

In [ ]:
# Analizar expensas (si existen)
df_exp = df_analisis[df_analisis['expensas'].notna() & (df_analisis['expensas'] > 0)]

print("=" * 60)
print("EXPENSAS (Gastos mensuales)")
print("=" * 60)

if len(df_exp) > 0:
    print(f"Propiedades con expensas registradas: {len(df_exp)}")
    print(f"Promedio de expensas: ${df_exp['expensas'].mean():,.0f} {df_exp['expensas_moneda'].mode()[0] if len(df_exp) > 0 else 'N/A'}")
    print(f"Mediana de expensas: ${df_exp['expensas'].median():,.0f}")
    print(f"Mínima: ${df_exp['expensas'].min():,.0f}")
    print(f"Máxima: ${df_exp['expensas'].max():,.0f}")
    
    # Ratio expensas/precio (cuánto representa mensualmente)
    df_exp['ratio_expensas_precio'] = (df_exp['expensas'] / df_exp['precio_usd']) * 100
    print(f"\nExpensas como % del precio (promedio): {df_exp['ratio_expensas_precio'].mean():.2f}%")
    print(f"  (Es decir, el 50% del precio en expensas mensuales = {df_exp['ratio_expensas_precio'].mean() / 12:.2f}% anual)")
else:
    print("No hay datos de expensas en el dataset.")

## 9. ANUNCIANTES: ¿Quién vende?

In [ ]:
# Tipo de anunciante
print("=" * 60)
print("TIPO DE ANUNCIANTE")
print("=" * 60)

por_anunciante = df_analisis['tipo_anunciante'].value_counts()
print(por_anunciante)

# Precios por tipo de anunciante
print("\n" + "="*60)
print("PRECIOS SEGÚN TIPO DE ANUNCIANTE")
print("="*60)

for anunciante in df_analisis['tipo_anunciante'].unique():
    datos = df_analisis[df_analisis['tipo_anunciante'] == anunciante]
    print(f"\n{anunciante.upper()}")
    print(f"  Cantidad: {len(datos)}")
    print(f"  Precio promedio: USD ${datos['precio_usd'].mean():,.0f}")
    print(f"  Precio promedio/m²: USD ${datos['precio_m2_calculado'].mean():,.0f}")

## 10. GRÁFICOS

In [ ]:
# Gráfico 1: Distribución de precios
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de precios
axes[0].hist(df_analisis['precio_usd'], bins=50, color='steelblue', edgecolor='black')
axes[0].axvline(df_analisis['precio_usd'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: ${df_analisis["precio_usd"].mean():,.0f}')
axes[0].axvline(df_analisis['precio_usd'].median(), color='green', linestyle='--', linewidth=2, label=f'Mediana: ${df_analisis["precio_usd"].median():,.0f}')
axes[0].set_xlabel('Precio (USD)', fontsize=11)
axes[0].set_ylabel('Cantidad de propiedades', fontsize=11)
axes[0].set_title('Distribución de Precios', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Box plot por tipo
df_analisis.boxplot(column='precio_usd', by='tipo', ax=axes[1])
axes[1].set_title('Precios por Tipo de Propiedad', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tipo', fontsize=11)
axes[1].set_ylabel('Precio (USD)', fontsize=11)
plt.suptitle('')  # Eliminar título redundante de matplotlib

plt.tight_layout()
plt.show()

print("✓ Gráfico de distribución creado")

In [ ]:
# Gráfico 2: Precio/m² por tipo
fig, ax = plt.subplots(figsize=(10, 5))

precio_m2_por_tipo = df_analisis.groupby('tipo')['precio_m2_calculado'].mean().sort_values(ascending=False)

precio_m2_por_tipo.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c', '#3498db'])
ax.set_title('Precio por Metro Cuadrado - Por Tipo de Propiedad', fontsize=12, fontweight='bold')
ax.set_xlabel('Tipo de Propiedad', fontsize=11)
ax.set_ylabel('Precio (USD/m²)', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Agregar valores en las barras
for i, v in enumerate(precio_m2_por_tipo):
    ax.text(i, v + 100, f'${v:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Gráfico de precio/m² creado")

In [ ]:
# Gráfico 3: Top 15 barrios más caros
fig, ax = plt.subplots(figsize=(12, 7))

top_15 = df_barrios.groupby('barrio')['precio_usd'].mean().nlargest(15).sort_values()

top_15.plot(kind='barh', ax=ax, color='coral')
ax.set_title('Top 15 Barrios Más Caros (Precio Promedio)', fontsize=12, fontweight='bold')
ax.set_xlabel('Precio Promedio (USD)', fontsize=11)
ax.set_ylabel('Barrio', fontsize=11)

# Agregar valores en las barras
for i, v in enumerate(top_15):
    ax.text(v + 20000, i, f'${v:,.0f}', va='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

print("✓ Gráfico de barrios creado")

In [ ]:
# Gráfico 4: Relación Ambientes vs Precio
fig, ax = plt.subplots(figsize=(10, 5))

df_amb_plot = df_amb[df_amb['ambientes'] <= 6]
precio_por_amb = df_amb_plot.groupby('ambientes')['precio_usd'].mean().sort_index()

precio_por_amb.plot(kind='bar', ax=ax, color='skyblue', edgecolor='navy')
ax.set_title('Precio Promedio por Cantidad de Ambientes', fontsize=12, fontweight='bold')
ax.set_xlabel('Número de Ambientes', fontsize=11)
ax.set_ylabel('Precio (USD)', fontsize=11)
ax.set_xticklabels([int(x) for x in precio_por_amb.index], rotation=0)

# Agregar valores en las barras
for i, v in enumerate(precio_por_amb):
    ax.text(i, v + 20000, f'${v:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Gráfico de ambientes creado")

## 11. RESUMEN EJECUTIVO

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║        RESUMEN EJECUTIVO: MERCADO INMOBILIARIO                ║
║        Capital Federal - Casas, Departamentos y PH            ║
╚════════════════════════════════════════════════════════════════╝

📊 DATASET
  • Total de propiedades analizadas: {:,}
  • Período de datos: 2026-05 a 2026-10
  • Moneda: USD

💰 PRECIOS
  • Promedio: USD ${:,.0f}
  • Mediana: USD ${:,.0f}
  • Rango: USD ${:,.0f} a ${:,.0f}
  • Dispersión: {:.1f}% (coef. variación)

📐 PRECIO POR METRO CUADRADO
  • Promedio: USD ${:,.0f}/m²
  • Mediana: USD ${:,.0f}/m²
  • Rango: USD ${:,.0f} a ${:,.0f}/m²

🏠 POR TIPO DE PROPIEDAD
  • DEPARTAMENTOS: {}: Precio promedio USD ${:,.0f}
  • CASAS: {}: Precio promedio USD ${:,.0f}
  • PH: {}: Precio promedio USD ${:,.0f}

📍 BARRIOS MÁS CAROS
  1. {} - USD ${:,.0f}
  2. {} - USD ${:,.0f}
  3. {} - USD ${:,.0f}

🏢 AMBIENTES COMUNES
  • Promedio de ambientes por propiedad: {:.1f}
  • Superficie promedio: {:.0f} m²
  • Relación: 1 ambiente = USD ${:,.0f}

👥 ANUNCIANTES
  • Inmobiliarias: {:,} anuncios ({:.1f}%)
  • Particulares: {:,} anuncios ({:.1f}%)

═══════════════════════════════════════════════════════════════
""".format(
    len(df_analisis),
    df_analisis['precio_usd'].mean(),
    df_analisis['precio_usd'].median(),
    df_analisis['precio_usd'].min(),
    df_analisis['precio_usd'].max(),
    cv,
    df_analisis['precio_m2_calculado'].mean(),
    df_analisis['precio_m2_calculado'].median(),
    df_analisis['precio_m2_calculado'].min(),
    df_analisis['precio_m2_calculado'].max(),
    len(df_analisis[df_analisis['tipo'] == 'departamento']),
    df_analisis[df_analisis['tipo'] == 'departamento']['precio_usd'].mean(),
    len(df_analisis[df_analisis['tipo'] == 'casa']),
    df_analisis[df_analisis['tipo'] == 'casa']['precio_usd'].mean(),
    len(df_analisis[df_analisis['tipo'] == 'ph']),
df_analisis[df_analisis['tipo'] == 'ph']['precio_usd'].mean(),
    top_15.index[-1], top_15.iloc[-1],
    top_15.index[-2], top_15.iloc[-2],
    top_15.index[-3], top_15.iloc[-3],
    df_amb['ambientes'].mean(),
    df_analisis['superficie_cubierta'].mean(),
    df_amb[df_amb['ambientes'] > 0].groupby('ambientes')['precio_usd'].mean().mean() / df_amb[df_amb['ambientes'] > 0]['ambientes'].mean(),
    len(df_analisis[df_analisis['tipo_anunciante'] == 'inmobiliaria']),
    len(df_analisis[df_analisis['tipo_anunciante'] == 'inmobiliaria']) / len(df_analisis) * 100,
    len(df_analisis[df_analisis['tipo_anunciante'] == 'particular']),
    len(df_analisis[df_analisis['tipo_anunciante'] == 'particular']) / len(df_analisis) * 100
))